In [1]:
import os
print('=== All input files ===')
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

=== All input files ===
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/predict.py
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/utils/prepare_clc_fce_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/utils/helpers.py
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/utils/preprocess_data.py
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/utils/__init__.py
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/models/mobilebert/model_best.th
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated/models/tinybert/model_best.th
/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/g

In [2]:
# ── Fixed Work Dir Setup ───────────────────────────────────────────────
import os, sys

# Correct dataset path (from your input files)
CODE_DATASET_PATH = '/kaggle/input/datasets/roseannnnnaguilar/gector-shared-updated/gector_shared_updated/gector_shared_updated'

# Model paths inside the input folder
TINYBERT_MODEL_PATH  = f'{CODE_DATASET_PATH}/models/tinybert/model_best.th'
ALBERT_MODEL_PATH    = f'{CODE_DATASET_PATH}/models/albert/model_best.th'
MOBILEBERT_MODEL_PATH = f'{CODE_DATASET_PATH}/models/mobilebert/model_best.th'

# BEA-2019 test/dev data
GOLD_M2   = '/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.dev.gold.bea19.m2'
TEST_ORIG = '/kaggle/input/datasets/roseannnnnaguilar/gec-test-data/gec_test_data/ABCN.test.bea19.orig'

# Output paths
WORK_DIR       = '/kaggle/working/gector'
DEV_ORIG       = '/kaggle/working/ABCN.dev.orig.txt'
DEV_PRED       = '/kaggle/working/dev_predictions.txt'
DEV_HYP_M2     = '/kaggle/working/dev_hyp.m2'
TEST_PRED      = '/kaggle/working/predictions.txt'
SUBMISSION_ZIP = '/kaggle/working/submission.zip'

# ── Prepare work dir ───────────────────────────────────────────────────
os.makedirs(WORK_DIR, exist_ok=True)
os.system(f'cp -r {CODE_DATASET_PATH}/* {WORK_DIR}/')  # copy all files to working dir
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Vocabulary path
VOCAB_PATH = f'{WORK_DIR}/data/output_vocabulary'

# ── Status Checks ──────────────────────────────────────────────────────
print('Work dir ready    :', os.listdir(WORK_DIR))
print('Vocab exists      :', os.path.isdir(VOCAB_PATH))
print('TinyBERT .th      :', os.path.isfile(f'{WORK_DIR}/models/tinybert/model_best.th'))
print('ALBERT .th        :', os.path.isfile(f'{WORK_DIR}/models/albert/model_best.th'))
print('MobileBERT .th    :', os.path.isfile(f'{WORK_DIR}/models/mobilebert/model_best.th'))
print('Dev gold m2       :', os.path.isfile(GOLD_M2))
print('Test orig         :', os.path.isfile(TEST_ORIG))

Work dir ready    : ['data', 'utils', 'models', 'predict.py', 'gector']
Vocab exists      : True
TinyBERT .th      : True
ALBERT .th        : True
MobileBERT .th    : True
Dev gold m2       : True
Test orig         : True


In [3]:
print('Installing ERRANT...')
os.system('pip install errant -q')
result = os.popen('errant_parallel --help 2>&1').read()
if 'usage' in result.lower() or 'optional' in result.lower():
    print('✅ ERRANT installed')
else:
    print('⚠️  Check ERRANT install:', result[:200])

Installing ERRANT...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.3/499.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.6 MB/s eta 0:00:00
✅ ERRANT installed


In [11]:
import torch, torch.nn as nn, subprocess, time

def print_model_params(model_obj, label):
    found = []
    if hasattr(model_obj, 'models') and isinstance(model_obj.models, (list, tuple)):
        for m in model_obj.models:
            if isinstance(m, nn.Module):
                found.append(m)
    if not found:
        for v in model_obj.__dict__.values():
            if isinstance(v, nn.Module):
                found.append(v)
            elif isinstance(v, (list, tuple)):
                for item in v:
                    if isinstance(item, nn.Module):
                        found.append(item)
    if not found:
        for attr in ('model', 'bert', 'encoder', 'gector', 'predictor'):
            if hasattr(model_obj, attr) and isinstance(getattr(model_obj, attr), nn.Module):
                found.append(getattr(model_obj, attr))
                break
    if not found:
        print(f'[WARN] Could not locate nn.Module inside {label}.')
        return 0, 0.0
    total = sum(p.numel() for mod in found for p in mod.parameters())
    size_mb = total * 4 / (1024 ** 2)
    line = '=' * 54
    print(line)
    print(f'MODEL PARAMETERS — {label}')
    print(line)
    print(f'  Total parameters : {total:,}')
    print(f'  Model size       : {size_mb:.1f} MB (float32)')
    print(line)
    return total, size_mb

def print_ensemble_params(param_list):
    line = '=' * 54
    print(line)
    print('ENSEMBLE MODEL PARAMETERS')
    print(line)
    for lbl, t, s in param_list:
        print(f'  {lbl:<12} : {t:>15,} params  ({s:>6.1f} MB)')
    print('-' * 54)
    grand = sum(t for _, t, _ in param_list)
    grand_s = sum(s for _, _, s in param_list)
    print(f'  {"TOTAL":<12} : {grand:>15,} params  ({grand_s:>6.1f} MB)')
    print(line)

def _gpu_smi():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total,memory.free',
             '--format=csv,noheader,nounits'], stderr=subprocess.DEVNULL
        ).decode().strip().split(',')
        return {'name': out[0].strip(), 'temp': out[1].strip(), 'util': out[2].strip(),
                'mem_used': int(out[3].strip()), 'mem_total': int(out[4].strip()), 'mem_free': int(out[5].strip())}
    except:
        return {}

def print_inference_stats(n_sentences, elapsed_sec, correction_counts, label='SET'):
    ms_per = (elapsed_sec / n_sentences * 1000) if n_sentences else 0
    alloc_mb  = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
    reserv_mb = torch.cuda.memory_reserved()  / 1024**2 if torch.cuda.is_available() else 0.0
    line = '=' * 54
    print(line)
    print(f'INFERENCE STATS — {label}')
    print(line)
    print(f'  Sentences        : {n_sentences}')
    print(f'  Total time       : {elapsed_sec:.1f} s ({elapsed_sec/60:.2f} min)')
    print(f'  Time / sentence  : {ms_per:.1f} ms')
    print()
    print(line)
    print('SENTENCES CORRECTED')
    print(line)
    for model, count in correction_counts.items():
        print(f'  {model:<12} : {count:>5} / {n_sentences}')
    g = _gpu_smi()
    if g:
        print()
        print(line)
        print('GPU USAGE (nvidia-smi)')
        print(line)
        print(f'  GPU 0          : {g["name"]}')
        print(f'  Temperature    : {g["temp"]} °C')
        print(f'  GPU Util       : {g["util"]} %')
        print(f'  Memory Util    : {round(g["mem_used"]/g["mem_total"]*100)} %')
        print(f'  Memory Used    : {g["mem_used"]} MB / {g["mem_total"]} MB')
        print(f'  Memory Free    : {g["mem_free"]} MB')
    print()
    print(line)
    print('TORCH CUDA MEMORY')
    print(line)
    print(f'  Allocated : {alloc_mb:.1f} MB')
    print(f'  Reserved  : {reserv_mb:.1f} MB')
    print(line)

_params_registry = []
print('✅ Stat helpers ready')

✅ Stat helpers ready


In [12]:
from gector.gec_model import GecBERTModel

print('Loading TinyBERT (4L-312D, uncased)...')
model_tinybert = GecBERTModel(
    vocab_path=VOCAB_PATH,
    model_paths=[TINYBERT_MODEL_PATH],
    max_len=50,
    min_len=3,
    iterations=5,
    min_error_probability=0.1,
    min_probability=0.0,
    lowercase_tokens=1,
    model_name='tinybert',
    special_tokens_fix=0,  # BERT-based: no offset fix
    log=False,
    confidence=0.0,
    is_ensemble=False,
    weigths=None
)
print('✅ TinyBERT ready!')

_t, _s = print_model_params(model_tinybert, 'TinyBERT')
_params_registry.append(('TinyBERT', _t, _s))

Loading TinyBERT (4L-312D, uncased)...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded 1 model(s) on cuda:0
✅ TinyBERT ready!
MODEL PARAMETERS — TinyBERT
  Total parameters : 15,917,126
  Model size       : 60.7 MB (float32)


In [13]:
print('Loading ALBERT-base-v2 (12L-768D)...')
model_albert = GecBERTModel(
    vocab_path=VOCAB_PATH,
    model_paths=[ALBERT_MODEL_PATH],
    max_len=50,
    min_len=3,
    iterations=5,
    min_error_probability=0.1,
    min_probability=0.0,
    lowercase_tokens=1,
    model_name='albert',
    special_tokens_fix=0,  # ALBERT: no special token offset fix
    log=False,
    confidence=0.0,
    is_ensemble=False,
    weigths=None
)
print('✅ ALBERT ready!')

_t, _s = print_model_params(model_albert, 'ALBERT')
_params_registry.append(('ALBERT', _t, _s))

Loading ALBERT-base-v2 (12L-768D)...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded 1 model(s) on cuda:0
✅ ALBERT ready!
MODEL PARAMETERS — ALBERT
  Total parameters : 15,533,198
  Model size       : 59.3 MB (float32)


In [14]:
print('Loading MobileBERT (uncased)...')
model_mobilebert = GecBERTModel(
    vocab_path=VOCAB_PATH,
    model_paths=[MOBILEBERT_MODEL_PATH],
    max_len=50,
    min_len=3,
    iterations=5,
    min_error_probability=0.1,
    min_probability=0.0,
    lowercase_tokens=1,
    model_name='mobilebert',
    special_tokens_fix=0,  # BERT-based: no offset fix
    log=False,
    confidence=0.0,
    is_ensemble=False,
    weigths=None
)
print('✅ MobileBERT ready!')

_t, _s = print_model_params(model_mobilebert, 'MobileBERT')
_params_registry.append(('MobileBERT', _t, _s))
print_ensemble_params(_params_registry)

Loading MobileBERT (uncased)...


Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertModel LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.dense.weight               | UNEXPECTED |  | 
mobilebert.embeddings.position_ids         | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded 1 model(s) on cuda:0
✅ MobileBERT ready!
MODEL PARAMETERS — MobileBERT
  Total parameters : 27,149,966
  Model size       : 103.6 MB (float32)
ENSEMBLE MODEL PARAMETERS
  TinyBERT     :      15,917,126 params  (  60.7 MB)
  ALBERT       :      15,533,198 params  (  59.3 MB)
  MobileBERT   :      27,149,966 params  ( 103.6 MB)
------------------------------------------------------
  TOTAL        :      58,600,290 params  ( 223.5 MB)


In [15]:
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def correct(paragraph: str, model) -> str:
    """Single sentence correction."""
    results, _ = model.handle_batch([paragraph])
    return results[0]

def ensemble(line: str) -> str:
    """
    3-way ensemble voting (line-level, one sentence per call).
    Voting cases:
      t == a == m  → all agree, use any
      t == a       → TinyBERT & ALBERT majority
      t == m       → TinyBERT & MobileBERT majority
      a == m       → ALBERT & MobileBERT majority
      all disagree → prefer MobileBERT (best test F0.5: 60.70)
    """
    t_out, _ = model_tinybert.handle_batch([line])
    a_out, _ = model_albert.handle_batch([line])
    m_out, _ = model_mobilebert.handle_batch([line])
    t, a, m = t_out[0], a_out[0], m_out[0]
    if t == a == m:
        return t
    elif t == a:
        return t
    elif t == m:
        return t
    elif a == m:
        return a
    else:
        return m  # all disagree → MobileBERT wins

print('✅ Inference helpers ready (3-way ensemble)')

✅ Inference helpers ready (3-way ensemble)


In [16]:
# Step 1: Count test sentences
with open(TEST_ORIG, encoding='utf-8') as f:
    test_lines = [l.rstrip('\n') for l in f]
test_count = len(test_lines)
print(f'Test sentences: {test_count}')

Test sentences: 4477


In [17]:
import time
from collections import Counter

# Step 2: Run ensemble on test
print(f'Running ensemble on {test_count} test sentences...')
model_times       = {'tinybert': 0.0, 'albert': 0.0, 'mobilebert': 0.0}
correction_counts = {'tinybert': 0, 'albert': 0, 'mobilebert': 0, 'ensemble': 0}

with open(TEST_PRED, 'w', encoding='utf-8') as f_out:
    for i, sentence in enumerate(test_lines):
        if not sentence.strip():
            f_out.write('\n')
            continue

        tokens = [sentence.strip().split()]

        # TinyBERT
        start = time.time()
        t_out, _ = model_tinybert.handle_batch(tokens)
        model_times['tinybert'] += time.time() - start

        # ALBERT
        start = time.time()
        a_out, _ = model_albert.handle_batch(tokens)
        model_times['albert'] += time.time() - start

        # MobileBERT
        start = time.time()
        m_out, _ = model_mobilebert.handle_batch(tokens)
        model_times['mobilebert'] += time.time() - start

        # Convert outputs to strings
        t = ' '.join(t_out[0])
        a = ' '.join(a_out[0])
        m = ' '.join(m_out[0])
        orig = sentence.strip()

        # Track per-model corrections
        if t != orig: correction_counts['tinybert']   += 1
        if a != orig: correction_counts['albert']     += 1
        if m != orig: correction_counts['mobilebert'] += 1

        # === Improved Ensemble Logic ===
        votes = [t, a, m]
        vote_count = Counter(votes)
        top_pred, top_count = vote_count.most_common(1)[0]

        # Majority voting
        if top_count >= 2:
            pred = top_pred
        else:
            pred = a  # fallback to ALBERT

        # Extra safety: ignore single-model corrections
        if pred != orig:
            changes = sum([t != orig, a != orig, m != orig])
            if changes == 1:
                pred = a

        if pred != orig:
            correction_counts['ensemble'] += 1

        f_out.write(pred + '\n')

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{test_count} done...')

# Verify line count
with open(TEST_PRED) as f:
    pred_count = sum(1 for _ in f)

print(f'✅ Test predictions done — {pred_count} lines written')
if pred_count != test_count:
    print(f'⚠️  Line count mismatch: input={test_count}, output={pred_count}')
else:
    print('✅ Line count matches test input')

elapsed_test = sum(model_times.values())
print_inference_stats(test_count, elapsed_test, correction_counts, label='TEST SET')

print()
print('=== Corrections Made ===')
for model, cnt in correction_counts.items():
    print(f'  {model:<12}: {cnt:>5} / {test_count} corrections')
print()
print('=== Inference Time ===')
for model in ['tinybert', 'albert', 'mobilebert']:
    print(f'  {model:<12}: {model_times[model]:.2f} sec')
print()
print('=== Avg Time per Sentence ===')
for model in ['tinybert', 'albert', 'mobilebert']:
    avg = model_times[model] / test_count if test_count > 0 else 0
    print(f'  {model:<12}: {avg:.6f} sec/sentence')

Running ensemble on 4477 test sentences...
  100/4477 done...
  200/4477 done...
  300/4477 done...
  400/4477 done...
  500/4477 done...
  600/4477 done...
  700/4477 done...
  800/4477 done...
  900/4477 done...
  1000/4477 done...
  1100/4477 done...
  1200/4477 done...
  1300/4477 done...
  1400/4477 done...
  1500/4477 done...
  1600/4477 done...
  1700/4477 done...
  1800/4477 done...
  1900/4477 done...
  2000/4477 done...
  2100/4477 done...
  2200/4477 done...
  2300/4477 done...
  2400/4477 done...
  2500/4477 done...
  2600/4477 done...
  2700/4477 done...
  2800/4477 done...
  2900/4477 done...
  3000/4477 done...
  3100/4477 done...
  3200/4477 done...
  3300/4477 done...
  3400/4477 done...
  3500/4477 done...
  3600/4477 done...
  3700/4477 done...
  3800/4477 done...
  3900/4477 done...
  4000/4477 done...
  4100/4477 done...
  4200/4477 done...
  4300/4477 done...
  4400/4477 done...
✅ Test predictions done — 4477 lines written
✅ Line count matches test input
INFERENCE

In [18]:
import zipfile, os

# ── Output zip paths ──────────────────────────────────────────────────
SUBMISSION_ZIP_ENSEMBLE   = '/kaggle/working/submission_ensemble.zip'
SUBMISSION_ZIP_TINYBERT   = '/kaggle/working/submission_tinybert.zip'
SUBMISSION_ZIP_ALBERT     = '/kaggle/working/submission_albert.zip'
SUBMISSION_ZIP_MOBILEBERT = '/kaggle/working/submission_mobilebert.zip'

# ── Individual prediction file paths ─────────────────────────────────
TINYBERT_PRED   = '/kaggle/working/predictions_tinybert.txt'
ALBERT_PRED     = '/kaggle/working/predictions_albert.txt'
MOBILEBERT_PRED = '/kaggle/working/predictions_mobilebert.txt'

# ── Write individual model predictions from the loop above ───────────
# (re-run inference to get per-model outputs separately)
print('Writing individual model prediction files...')

t_preds, a_preds, m_preds = [], [], []

for sentence in test_lines:
    if not sentence.strip():
        t_preds.append('')
        a_preds.append('')
        m_preds.append('')
        continue
    tokens = [sentence.strip().split()]
    t_out, _ = model_tinybert.handle_batch(tokens)
    a_out, _ = model_albert.handle_batch(tokens)
    m_out, _ = model_mobilebert.handle_batch(tokens)
    t_preds.append(' '.join(t_out[0]))
    a_preds.append(' '.join(a_out[0]))
    m_preds.append(' '.join(m_out[0]))

with open(TINYBERT_PRED,   'w', encoding='utf-8') as f:
    f.write('\n'.join(t_preds) + '\n')
with open(ALBERT_PRED,     'w', encoding='utf-8') as f:
    f.write('\n'.join(a_preds) + '\n')
with open(MOBILEBERT_PRED, 'w', encoding='utf-8') as f:
    f.write('\n'.join(m_preds) + '\n')

print('✅ Individual prediction files written')

# ── Pack all zips ─────────────────────────────────────────────────────
zips = [
    (SUBMISSION_ZIP_ENSEMBLE,   TEST_PRED),        # already written by loop above
    (SUBMISSION_ZIP_TINYBERT,   TINYBERT_PRED),
    (SUBMISSION_ZIP_ALBERT,     ALBERT_PRED),
    (SUBMISSION_ZIP_MOBILEBERT, MOBILEBERT_PRED),
]

print('\n=== Submission Zips ===')
for zip_path, pred_path in zips:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname='predictions.txt')
    size_kb = os.path.getsize(zip_path) / 1024
    name    = os.path.basename(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        contents = zf.namelist()
    print(f'  ✅ {name:<35} {size_kb:.1f} KB  {contents}')

Writing individual model prediction files...
✅ Individual prediction files written

=== Submission Zips ===
  ✅ submission_ensemble.zip             155.9 KB  ['predictions.txt']
  ✅ submission_tinybert.zip             155.9 KB  ['predictions.txt']
  ✅ submission_albert.zip               155.9 KB  ['predictions.txt']
  ✅ submission_mobilebert.zip           155.9 KB  ['predictions.txt']


In [19]:
# Step 1: Extract source sentences from gold m2
print('Extracting dev orig from gold m2...')
with open(GOLD_M2, encoding='utf-8') as f_in, \
     open(DEV_ORIG, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        if line.startswith('S '):
            f_out.write(line[2:])

with open(DEV_ORIG) as f:
    count = sum(1 for _ in f)
print(f'✅ Extracted {count} source sentences')

Extracting dev orig from gold m2...
✅ Extracted 4384 source sentences


In [20]:
import time
from collections import Counter

# Step 2: Run ensemble predictions on dev
print(f'Running ensemble on {count} dev sentences...')
model_times       = {'tinybert': 0.0, 'albert': 0.0, 'mobilebert': 0.0}
correction_counts = {'tinybert': 0, 'albert': 0, 'mobilebert': 0, 'ensemble': 0}

with open(DEV_ORIG, encoding='utf-8') as f_in, \
     open(DEV_PRED, 'w', encoding='utf-8') as f_out:

    for i, line in enumerate(f_in):
        sentence = line.strip()

        if not sentence:
            f_out.write('\n')
            continue

        tokens = [sentence.split()]

        # TinyBERT
        start = time.time()
        t_out, _ = model_tinybert.handle_batch(tokens)
        model_times['tinybert'] += time.time() - start

        # ALBERT
        start = time.time()
        a_out, _ = model_albert.handle_batch(tokens)
        model_times['albert'] += time.time() - start

        # MobileBERT
        start = time.time()
        m_out, _ = model_mobilebert.handle_batch(tokens)
        model_times['mobilebert'] += time.time() - start

        # Convert outputs to strings
        t = ' '.join(t_out[0])
        a = ' '.join(a_out[0])
        m = ' '.join(m_out[0])
        orig = sentence

        # Track per-model corrections
        if t != orig: correction_counts['tinybert']   += 1
        if a != orig: correction_counts['albert']     += 1
        if m != orig: correction_counts['mobilebert'] += 1

        # === Improved Ensemble Logic ===
        votes = [t, a, m]
        vote_count = Counter(votes)
        top_pred, top_count = vote_count.most_common(1)[0]

        # Majority voting
        if top_count >= 2:
            pred = top_pred
        else:
            pred = a  # fallback to ALBERT

        # Extra safety: ignore single-model corrections
        if pred != orig:
            changes = sum([t != orig, a != orig, m != orig])
            if changes == 1:
                pred = a

        if pred != orig:
            correction_counts['ensemble'] += 1

        f_out.write(pred + '\n')

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{count} done...')

print(f'✅ Dev predictions written to {DEV_PRED}')

elapsed_dev = sum(model_times.values())
print_inference_stats(count, elapsed_dev, correction_counts, label='DEV SET')

print()
print('=== Corrections Made ===')
for model, cnt in correction_counts.items():
    print(f'  {model:<12}: {cnt:>5} / {count} corrections')
print()
print('=== Inference Time ===')
for model in ['tinybert', 'albert', 'mobilebert']:
    print(f'  {model:<12}: {model_times[model]:.2f} sec')
print()
print('=== Avg Time per Sentence ===')
for model in ['tinybert', 'albert', 'mobilebert']:
    avg = model_times[model] / count if count > 0 else 0
    print(f'  {model:<12}: {avg:.6f} sec/sentence')

Running ensemble on 4384 dev sentences...
  100/4384 done...
  200/4384 done...
  300/4384 done...
  400/4384 done...
  500/4384 done...
  600/4384 done...
  700/4384 done...
  800/4384 done...
  900/4384 done...
  1000/4384 done...
  1100/4384 done...
  1200/4384 done...
  1300/4384 done...
  1400/4384 done...
  1500/4384 done...
  1600/4384 done...
  1700/4384 done...
  1800/4384 done...
  1900/4384 done...
  2000/4384 done...
  2100/4384 done...
  2200/4384 done...
  2300/4384 done...
  2400/4384 done...
  2500/4384 done...
  2600/4384 done...
  2700/4384 done...
  2800/4384 done...
  2900/4384 done...
  3000/4384 done...
  3100/4384 done...
  3200/4384 done...
  3300/4384 done...
  3400/4384 done...
  3500/4384 done...
  3600/4384 done...
  3700/4384 done...
  3800/4384 done...
  3900/4384 done...
  4000/4384 done...
  4100/4384 done...
  4200/4384 done...
  4300/4384 done...
✅ Dev predictions written to /kaggle/working/dev_predictions.txt
INFERENCE STATS — DEV SET
  Sentences     

In [21]:
# Step 3: ERRANT annotation
print('Running ERRANT annotation on dev predictions...')
os.system(f'errant_parallel -orig {DEV_ORIG} -cor {DEV_PRED} -out {DEV_HYP_M2}')

if os.path.exists(DEV_HYP_M2):
    print('✅ dev_hyp.m2 created!')
else:
    print('❌ dev_hyp.m2 not created')

# Verify sentence counts match
with open(DEV_HYP_M2) as f:
    hyp_count = sum(1 for line in f if line.startswith('S '))
with open(GOLD_M2) as f:
    gold_count = sum(1 for line in f if line.startswith('S '))
print(f'hyp sentences:  {hyp_count}')
print(f'gold sentences: {gold_count}')

if hyp_count != gold_count:
    print('❌ Sentence count mismatch!')
else:
    print('\n========================================')
    print('ERRANT Scores on BEA-2019 Dev Set:')
    print('(Precision / Recall / F0.5)')
    print('========================================')
    result = os.popen(f'errant_compare -hyp {DEV_HYP_M2} -ref {GOLD_M2} 2>&1').read()
    print(result)

Running ERRANT annotation on dev predictions...
Loading resources...
Processing parallel files...
✅ dev_hyp.m2 created!
hyp sentences:  4384
gold sentences: 4384

ERRANT Scores on BEA-2019 Dev Set:
(Precision / Recall / F0.5)

=========== Span-Based Correction ============
TP	FP	FN	Prec	Rec	F0.5
1606	1322	5855	0.5485	0.2153	0.4188


